In [7]:
%%time

import pandas as pd
import re
from transformers import pipeline, AutoTokenizer, logging
logging.set_verbosity_error()

# ---------- 0. Cargar el dataset ----------
df = pd.read_pickle('Dataset_final.pkl')

# ---------- 0.1 Excluir películas no válidas para análisis en inglés ----------
peliculas_excluidas = ['Talk to Her', 'Anatomy of a Fall']
df = df[~df['Title'].isin(peliculas_excluidas)].reset_index(drop=True)

# ---------- 1. Configuración del modelo ----------
MODEL_NAME = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"
MAX_TOKENS = 512

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
sentiment_analyzer = pipeline(
    "sentiment-analysis",
    model=MODEL_NAME,
    tokenizer=tokenizer,
    device='cuda'
)

# ---------- 2. Chunking por oraciones (no corta frases a la mitad) ----------

def split_sentences(texto):
    oraciones = re.split(r'(?<=[.!?])\s+', texto.strip())
    return [o for o in oraciones if o.strip() != ""]

def chunk_by_sentences(texto, max_tokens=MAX_TOKENS - 2):
    oraciones = split_sentences(texto)
    chunks = []
    chunk_actual, tokens_actual = [], 0

    for oracion in oraciones:
        n_tokens = len(tokenizer.encode(oracion, add_special_tokens=False))

        if n_tokens > max_tokens:
            if chunk_actual:
                chunks.append(" ".join(chunk_actual))
                chunk_actual, tokens_actual = [], 0
            chunks.append(oracion)  # oración extrema, se truncará al analizarla
            continue

        if tokens_actual + n_tokens > max_tokens:
            chunks.append(" ".join(chunk_actual))
            chunk_actual, tokens_actual = [oracion], n_tokens
        else:
            chunk_actual.append(oracion)
            tokens_actual += n_tokens

    if chunk_actual:
        chunks.append(" ".join(chunk_actual))

    return chunks

# ---------- 3. Análisis con media ponderada por tokens ----------

def analyze_long_text(texto):
    if not texto or not isinstance(texto, str) or texto.strip() == "":
        return None, None, 0

    chunks = chunk_by_sentences(texto)
    salidas = sentiment_analyzer(chunks, batch_size=8, truncation=True)

    weighted_sum, total_tokens = 0, 0
    for chunk, salida in zip(chunks, salidas):
        n_tokens = len(tokenizer.encode(chunk, add_special_tokens=False))
        signed_score = salida['score'] if salida['label'] == 'POSITIVE' else -salida['score']
        weighted_sum += signed_score * n_tokens
        total_tokens += n_tokens

    promedio = weighted_sum / total_tokens
    label = 'POSITIVE' if promedio >= 0 else 'NEGATIVE'
    return promedio, label, len(chunks)

# ---------- 4. Aplicar a cada personaje dentro de un Script_Dict ----------

def analyze_script_dict(script_dict):
    """
    Recibe {personaje: texto} y devuelve tres dicts:
    {personaje: score}, {personaje: label}, {personaje: n_chunks}
    """
    scores, labels, n_chunks_dict = {}, {}, {}
    for personaje, texto in script_dict.items():
        score, label, n_chunks = analyze_long_text(texto)
        scores[personaje] = score
        labels[personaje] = label
        n_chunks_dict[personaje] = n_chunks
    return scores, labels, n_chunks_dict

# ---------- 5. Aplicar a todo el dataset y crear las columnas nuevas ----------

resultados = df['Script_Dict'].apply(analyze_script_dict)

df['Distilbert_Sentiment_Score'] = resultados.apply(lambda x: x[0])
df['Distilbert_Sentiment_Label'] = resultados.apply(lambda x: x[1])
df['Distilbert_N_Chunks'] = resultados.apply(lambda x: x[2])

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 8020.14it/s]


CPU times: total: 49.5 s
Wall time: 51.5 s


In [9]:
df.to_pickle('Dataset_final_NLP.pkl')

In [8]:
df

,IMDb_ID,Title,Oscar_Year,Spoken_Languages,Rating_Score,Votes_Count,Runtime_Min,Budget,Revenue,Genres,...,Unknown_Characters_Count,Words_Male,Words_Female,Words_Unknown,AverageWords_male,AverageWords_female,AverageWords_unknown,Distilbert_Sentiment_Score,Distilbert_Sentiment_Label,Distilbert_N_Chunks
0,tt0167260,The Lord of the Rings: The Return of the King,2004,English,8.500,26611,201,94000000,1118888979,"Adventure, Fantasy, Action",...,0,6274,353,0,153.02,117.67,0.00,"{'ARAGORN': -0.4162187962918668, 'ARWEN': -0.8...","{'ARAGORN': 'NEGATIVE', 'ARWEN': 'NEGATIVE', '...","{'ARAGORN': 2, 'ARWEN': 1, 'BILBO': 1, 'DENETH..."
1,tt0169547,American Beauty,2000,English,8.000,13058,122,15000000,356296601,Drama,...,4,4636,3287,113,515.11,298.82,28.25,"{'ANGELA': 0.22488881396346314, 'ANNOUNCER': 0...","{'ANGELA': 'POSITIVE', 'ANNOUNCER': 'POSITIVE'...","{'ANGELA': 3, 'ANNOUNCER': 1, 'ATTENDANT': 1, ..."
2,tt0172495,Gladiator,2001,English,8.200,20925,155,103000000,465516248,"Action, Drama, Adventure",...,3,7664,1174,27,232.24,587.00,9.00,"{'ASSASSIN #1': 0.987301230430603, 'ASSASSIN #...","{'ASSASSIN #1': 'POSITIVE', 'ASSASSIN #2': 'PO...","{'ASSASSIN #1': 1, 'ASSASSIN #2': 1, 'ASSASSIN..."
3,tt0268978,A Beautiful Mind,2002,English,7.856,11093,135,58000000,316791257,"Drama, Romance",...,1,7333,1419,13,282.04,354.75,13.00,"{'ADJUNCT': -0.9991183876991272, 'ALICIA': -0....","{'ADJUNCT': 'NEGATIVE', 'ALICIA': 'NEGATIVE', ...","{'ADJUNCT': 1, 'ALICIA': 4, 'ANALYST': 1, 'ARC..."
4,tt0299658,Chicago,2003,"Hungarian, English",7.131,2912,113,45000000,306776732,"Comedy, Crime, Drama",...,3,4323,4613,202,254.29,307.53,67.33,"{'AMOS': -0.9966522327776952, 'ANNIE': -0.9906...","{'AMOS': 'NEGATIVE', 'ANNIE': 'NEGATIVE', 'BAI...","{'AMOS': 3, 'ANNIE': 1, 'BAILIFF': 1, 'BANDLEA..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70,tt10272386,The Father,2021,English,8.100,3503,97,6000000,21029340,Drama,...,0,4669,2731,0,1167.25,682.75,0.00,"{'ANNE': -0.4915123784858169, 'ANTHONY': -0.33...","{'ANNE': 'NEGATIVE', 'ANTHONY': 'NEGATIVE', 'D...","{'ANNE': 6, 'ANTHONY': 12, 'DOCTOR': 1, 'LAURA..."
71,tt10366460,CODA,2022,"English, Spanish",7.900,2469,112,10000000,1905058,"Drama, Music, Romance",...,3,3387,2439,102,153.95,152.44,34.00,"{'ADELE GIRL': 0.9998691082000732, 'ANGELA': -...","{'ADELE GIRL': 'POSITIVE', 'ANGELA': 'NEGATIVE...","{'ADELE GIRL': 1, 'ANGELA': 1, 'ARTHUR': 1, 'A..."
72,tt13669038,Women Talking,2023,English,6.862,680,104,20000000,7589419,Drama,...,0,1621,7884,0,180.11,525.60,0.00,"{'AGATA': 0.4849160841999165, 'ANNA': -0.83144...","{'AGATA': 'POSITIVE', 'ANNA': 'NEGATIVE', 'AUG...","{'AGATA': 5, 'ANNA': 1, 'AUGUST': 4, 'AUTJE': ..."
73,tt23561236,American Fiction,2024,English,7.272,1244,117,16000000,22483370,"Comedy, Drama",...,0,7743,2872,0,267.00,205.14,0.00,"{'AGNES': 0.7623928189277649, 'AILENE': 0.9867...","{'AGNES': 'POSITIVE', 'AILENE': 'POSITIVE', 'A...","{'AGNES': 1, 'AILENE': 1, 'ALVIN': 1, 'ARTHUR'..."


## Flatten version

In [10]:
%%time

import pandas as pd
import re
from transformers import pipeline, AutoTokenizer, logging
logging.set_verbosity_error()

# ---------- 0. Cargar el dataset ----------
df_f = pd.read_pickle('Dataset_final.pkl')

# ---------- 0.1 Excluir películas no válidas para análisis en inglés ----------
peliculas_excluidas = ['Talk to Her', 'Anatomy of a Fall']
df_f = df_f[~df_f['Title'].isin(peliculas_excluidas)].reset_index(drop=True)

# ---------- 1. Configuración del modelo ----------
MODEL_NAME = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"
MAX_TOKENS = 512

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
sentiment_analyzer = pipeline(
    "sentiment-analysis",
    model=MODEL_NAME,
    tokenizer=tokenizer,
    device='cuda'
)

# ---------- 2. Chunking por oraciones (no corta frases a la mitad) ----------

def split_sentences(texto):
    oraciones = re.split(r'(?<=[.!?])\s+', texto.strip())
    return [o for o in oraciones if o.strip() != ""]

def chunk_by_sentences(texto, max_tokens=MAX_TOKENS - 2):
    oraciones = split_sentences(texto)
    chunks = []
    chunk_actual, tokens_actual = [], 0

    for oracion in oraciones:
        n_tokens = len(tokenizer.encode(oracion, add_special_tokens=False))

        if n_tokens > max_tokens:
            if chunk_actual:
                chunks.append(" ".join(chunk_actual))
                chunk_actual, tokens_actual = [], 0
            chunks.append(oracion)
            continue

        if tokens_actual + n_tokens > max_tokens:
            chunks.append(" ".join(chunk_actual))
            chunk_actual, tokens_actual = [oracion], n_tokens
        else:
            chunk_actual.append(oracion)
            tokens_actual += n_tokens

    if chunk_actual:
        chunks.append(" ".join(chunk_actual))

    return chunks

# ---------- 3. Análisis con media ponderada por tokens ----------

def analyze_long_text(texto):
    if not texto or not isinstance(texto, str) or texto.strip() == "":
        return None, None, 0

    chunks = chunk_by_sentences(texto)
    salidas = sentiment_analyzer(chunks, batch_size=8, truncation=True)

    weighted_sum, total_tokens = 0, 0
    for chunk, salida in zip(chunks, salidas):
        n_tokens = len(tokenizer.encode(chunk, add_special_tokens=False))
        signed_score = salida['score'] if salida['label'] == 'POSITIVE' else -salida['score']
        weighted_sum += signed_score * n_tokens
        total_tokens += n_tokens

    promedio = weighted_sum / total_tokens
    label = 'POSITIVE' if promedio >= 0 else 'NEGATIVE'
    return promedio, label, len(chunks)

# ---------- 4. Aplicar a cada personaje dentro de un Script_Dict ----------

def analyze_script_dict(script_dict):
    scores, labels, n_chunks_dict = {}, {}, {}
    for personaje, texto in script_dict.items():
        score, label, n_chunks = analyze_long_text(texto)
        scores[personaje] = score
        labels[personaje] = label
        n_chunks_dict[personaje] = n_chunks
    return scores, labels, n_chunks_dict

# ---------- 5. Aplicar a todo el dataset ----------

resultados = df_f['Script_Dict'].apply(analyze_script_dict)

df_f['Distilbert_Sentiment_Score'] = resultados.apply(lambda x: x[0])
df_f['Distilbert_Sentiment_Label'] = resultados.apply(lambda x: x[1])
df_f['Distilbert_N_Chunks'] = resultados.apply(lambda x: x[2])

# ---------- 6. Guardar el dataset actualizado (opcional) ----------

df_f.to_pickle('Dataset_final_NLP_flatten.pkl')

# ---------- 7. Versión aplanada: una fila por personaje ----------

filas = []
for idx, row in df_f.iterrows():
    for personaje in row['Script_Dict'].keys():
        filas.append({
            'IMDb_ID': row['IMDb_ID'],
            'Title': row['Title'],
            'Award': row['Award'],
            'Character': personaje,
            'Gender': row['Characters_Genders'].get(personaje, 'unknown'),
            'Sentiment_Score': row['Distilbert_Sentiment_Score'].get(personaje),
            'Sentiment_Label': row['Distilbert_Sentiment_Label'].get(personaje),
            'N_Chunks': row['Distilbert_N_Chunks'].get(personaje)
        })

df_sentiment = pd.DataFrame(filas)

# ---------- 8. Análisis por género y categoría de Oscar ----------

resumen = df_sentiment.groupby(['Gender', 'Award'])['Sentiment_Score'].agg(['mean', 'std', 'count'])
print(resumen)

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 7447.88it/s]


                                 mean       std  count
Gender  Award                                         
female  Adapted Screenplay  -0.205433  0.916861    312
        Best Picture        -0.229743  0.898990    238
        Original Screenplay -0.221722  0.900527    287
male    Adapted Screenplay  -0.371374  0.869766    821
        Best Picture        -0.327533  0.884578    754
        Original Screenplay -0.263742  0.888898    596
unknown Adapted Screenplay   0.059721  0.985571     32
        Best Picture        -0.216694  0.928579     36
        Original Screenplay -0.240360  0.947532     43
CPU times: total: 46.9 s
Wall time: 48.7 s
